In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../resources/full_dataset.csv')
df.head()

,V020,S111A,V501,HV104,SB267,SB236,SB240,WBP24,WBP25,WBP16,...,V744A,V744B,V744C,V744D,V744E,CASEID,V001,V005,V021,V022
0,1,1,1,2,139.0,0.0,NaN,129.0,97.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1 1 3,1,156207,1,1
1,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 4 2,1,156207,1,1
2,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1 7 3,1,156207,1,1
3,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 10 2,1,156207,1,1
4,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 13 2,1,156207,1,1


In [3]:
from tools.recode import recode, clean_recode, parse_recode, parse_all_recodes
from tools.recode import binary_cols, nominal_cols, ordinal_cols, numerical_continuous_cols, numerical_discrete_cols
from tools.recode import domain_groups

parse_all_recodes()

Code,Category Label
1,Ever-married
0,All woman
Code,Category Label
1,Currently married
2,Separated
3,Deserted
4,Divorced
5,Widowed
Code,Category Label
0,Never married


In [4]:
def parse(var: str) -> None:
    """
    Details of a variable (displayed simply)
    """
    var_dict = recode.get(var, None) or clean_recode.get(var, None)
    if not var_dict:
        print("Variable not found in the recode list!")
        return None

    print(f"Name: {var_dict['name']}")
    value = var_dict.get("value", None)
    if value is None:
        print("Value: Continuous variable")
        return None
    print(f"Categories: ")
    for i, j in value.items():
        print(f"\t{i}: {j}")

## Data Cleaning

#### Selection of Ever-married Women

In [5]:
# Check values for ever-married women
display(df["V020"].value_counts())
print("-"*40)
display(df["HV104"].value_counts())

V020
1    30078
Name: count, dtype: int64

----------------------------------------


HV104
2    30078
Name: count, dtype: int64

In [6]:
# Check values for ever-married women
parse("V020")
print("-"*40)
parse("HV104")

Name: Type of sample or ever-married indicator
Categories: 
	1: Ever-married
	0: All woman
----------------------------------------
Name: Sex of household member
Categories: 
	1: Male
	2: Female
	9: Missing


In [7]:
# Filter by "Ever married"
df = df[df["V020"] == 1]

# Filter by "Gender"
df = df[df["HV104"] == 2]

df.shape

(30078, 68)

```markdown
Because the Individual Record (IR file, BDIR81FL) and Personal Record (PR file, BDPR81FL) were merged on cluster-household-line number, the merged file initially contained the biomarker records of all household members captured in the Personal Record, not exclusively the eligible female respondent. Restricting to HV104 = 2 (female) in addition to V020 = 1 (ever-married) therefore removes co-resident household members' records introduced by the merge, rather than performing a redundant sex restriction.
```

#### Exploring `Exposure`, `Outcome` and `Effect modifier`

In [8]:
# Check values of Diabetes
for i in domain_groups.get("diabetes"):
    display(df[i].value_counts())
    print("-"*40)

SB267
100.0    182
98.0     171
96.0     167
99.0     167
103.0    166
        ... 
227.0      1
159.0      1
242.0      1
346.0      1
233.0      1
Name: count, Length: 201, dtype: int64

----------------------------------------


SB236
0.0    4908
1.0     229
9.0      51
Name: count, dtype: int64

----------------------------------------


SB240
1.0    161
0.0     68
Name: count, dtype: int64

----------------------------------------


In [9]:
# Check values of Diabetes
for i in domain_groups.get("diabetes"):
    parse(i)
    print()
    print("-"*40)

Name: Plasma glucose (mg/dL)
Value: Continuous variable

----------------------------------------
Name: Ever diagnosed with diabetes
Categories: 
	0: No
	1: Yes
	9: Missing

----------------------------------------
Name: Currently taking medication for diabetes
Categories: 
	0: No
	1: Yes

----------------------------------------


In [10]:
# Replace missing values with NaN
df["SB236"] = df["SB236"].replace(9, np.nan)
del recode["SB236"]["value"][9]
df["SB236"].value_counts()

SB236
0.0    4908
1.0     229
Name: count, dtype: int64

In [11]:
# Check values of Hypertension
for i in domain_groups.get("hypertension"):
    display(df[i].value_counts())
    print("-"*40)

WBP24
109.0    188
105.0    175
106.0    171
111.0    170
114.0    162
        ... 
212.0      1
234.0      1
198.0      1
239.0      1
219.0      1
Name: count, Length: 118, dtype: int64

----------------------------------------


WBP25
75.0     229
72.0     225
73.0     217
76.0     216
78.0     209
        ... 
126.0      1
48.0       1
127.0      1
135.0      1
124.0      1
Name: count, Length: 77, dtype: int64

----------------------------------------


WBP16
0.0    4603
1.0     534
Name: count, dtype: int64

----------------------------------------


WBP19
1.0    343
0.0    191
Name: count, dtype: int64

----------------------------------------


In [12]:
# Check values of Hypertension
for i in domain_groups.get("hypertension"):
    parse(i)
    print("-"*40)

Name: Final systolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Final diastolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Previously diagnosed with hypertension
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Currently taking blood pressure medication
Categories: 
	0: No
	1: Yes
----------------------------------------


In [13]:
# Check values of Obesity
for i in domain_groups.get("obesity"):
    display(df[i].value_counts())
    print("-"*40)

HA40
9999.0    107
2605.0     21
2190.0     19
2245.0     19
2304.0     18
         ... 
3131.0      1
1529.0      1
1424.0      1
1612.0      1
1679.0      1
Name: count, Length: 1945, dtype: int64

----------------------------------------


In [14]:
# Check values of Obesity
for i in domain_groups.get("obesity"):
    parse(i)
    print("-"*40)

Name: Body Mass Index (BMI)
Value: Continuous variable
----------------------------------------


In [15]:
# Check values of Wealth
for i in domain_groups.get("ses_wealth"):
    display(df[i].value_counts())
    print("-"*40)

V190
5    6570
4    6168
3    5926
2    5855
1    5559
Name: count, dtype: int64

----------------------------------------


V190A
1    6122
2    6041
5    6000
3    5969
4    5946
Name: count, dtype: int64

----------------------------------------


V191
 149923    31
 165645    29
 145596    28
 170159    22
 169972    21
           ..
-96439      1
-80242      1
-66628      1
-98932      1
-53408      1
Name: count, Length: 21815, dtype: int64

----------------------------------------


V191A
 87686     31
 103076    29
 83451     28
 107495    22
 107311    21
           ..
-78579      1
-58673      1
-41942      1
-81643      1
-25695      1
Name: count, Length: 21865, dtype: int64

----------------------------------------


In [16]:
# Check values of Wealth
for i in domain_groups.get("ses_wealth"):
    parse(i)
    print("-"*40)

Name: Wealth index combined (categorical)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest
----------------------------------------
Name: Wealth index combined (categorical, urban/rural clustered)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest
----------------------------------------
Name: Wealth index factor score combined
Value: Continuous variable
----------------------------------------
Name: Wealth index factor score combined (urban/rural clustered)
Value: Continuous variable
----------------------------------------


In [17]:
# Check values of Mental health variable
for i in domain_groups.get("mental_health"):
    display(df[i].value_counts())
    print("-"*40)

MTH22
0.0     4504
2.0     2577
3.0     2480
1.0     2477
4.0     2115
5.0     1650
6.0     1175
7.0      807
8.0      613
9.0      563
10.0     253
11.0     199
12.0     125
13.0     105
14.0      80
15.0      69
16.0      46
18.0      42
17.0      34
19.0      24
21.0      15
20.0       9
22.0       9
23.0       5
24.0       4
25.0       3
27.0       2
26.0       2
Name: count, dtype: int64

----------------------------------------


MTH24
0.0     4590
1.0     3017
2.0     2839
3.0     2400
4.0     1765
5.0     1451
6.0     1224
7.0      961
8.0      525
9.0      319
10.0     221
11.0     186
12.0     130
14.0      85
13.0      74
15.0      70
16.0      40
17.0      31
18.0      20
21.0      16
19.0      15
20.0       8
Name: count, dtype: int64

----------------------------------------


In [18]:
print(domain_groups.get("diabetes"))
print(domain_groups.get("hypertension"))
print(domain_groups.get("obesity"))
print(domain_groups.get("ses_wealth"))
print(domain_groups.get("mental_health"))

['SB267', 'SB236', 'SB240']
['WBP24', 'WBP25', 'WBP16', 'WBP19']
['HA40']
['V190', 'V190A', 'V191', 'V191A']
['MTH22', 'MTH24']


In [19]:
print("Missing vlaue for Diabetes")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Hypertension")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Obesity")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Socioeconomic Status")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Depression")
print("\tMissing:", df.shape[0] - df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Anxiety")
print("\tMissing:", df.shape[0] - df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")

Missing vlaue for Diabetes
	Missing: 24890
	Available: 5188
	Missing: 82.75151273355941 %

Missing vlaue for Hypertension
	Missing: 24941
	Available: 5137


	Missing: 82.92107187977925 %

Missing vlaue for Obesity
	Missing: 20025
	Available: 10053
	Missing: 66.5769000598444 %

Missing vlaue for Socioeconomic Status
	Missing: 0
	Available: 30078
	Missing: 0.0 %

Missing vlaue for Depression
	Missing: 10091
	Available: 19987
	Missing: 33.54943812753507 %

Missing vlaue for Anxiety
	Missing: 10091
	Available: 19987
	Missing: 33.54943812753507 %


In [20]:
# Filter missing values for Cardiometabolic Burden
df_new = df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3]
df_new = df_new.iloc[df_new[domain_groups.get("hypertension")].isna().sum(axis=1) < 4]
df_new = df_new.iloc[df_new[domain_groups.get("obesity")].isna().sum(axis=1) < 1]

# Filter missing values for SES
df_new = df_new.iloc[df_new[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4]

# Filter missing vlaues for Depression and Anxiety
df_depression = df_new.iloc[df_new[["MTH22"]].isna().sum(axis=1) < 1]
df_anxiety = df_new.iloc[df_new[["MTH24"]].isna().sum(axis=1) < 1]

print(df_depression.shape, df_anxiety.shape)

(5137, 68) (5137, 68)


## Cardiometabolic Burden

#### Diabetes Variable

In [21]:
for i in domain_groups.get("diabetes"):
    parse(i)
    print("-"*40)

Name: Plasma glucose (mg/dL)
Value: Continuous variable
----------------------------------------
Name: Ever diagnosed with diabetes
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Currently taking medication for diabetes
Categories: 
	0: No
	1: Yes
----------------------------------------


In [22]:
df_anxiety["Diabetes"] = ((df_anxiety["SB267"] >= 126) | (df_anxiety["SB236"] == 1) | (df_anxiety["SB240"] == 1)).astype(int)
df_depression["Diabetes"] = ((df_depression["SB267"] >= 126) | (df_depression["SB236"] == 1) | (df_depression["SB240"] == 1)).astype(int)

display(df_anxiety["Diabetes"].value_counts())
print("-"*40)
display(df_depression["Diabetes"].value_counts())

Diabetes
0    4612
1     525
Name: count, dtype: int64

----------------------------------------


Diabetes
0    4612
1     525
Name: count, dtype: int64

#### Hypertension Variable

In [23]:
for i in domain_groups.get("hypertension"):
    parse(i)
    print("-"*40)

Name: Final systolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Final diastolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Previously diagnosed with hypertension
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Currently taking blood pressure medication
Categories: 
	0: No
	1: Yes
----------------------------------------


In [24]:
df_anxiety["Hypertension"] = ((df_anxiety["WBP24"] >= 140) | (df_anxiety["WBP25"] >= 90) | (df_anxiety["WBP16"] == 1) | (df_anxiety["WBP19"] == 1)).astype(int)
df_depression["Hypertension"] = ((df_depression["WBP24"] >= 140) | (df_depression["WBP25"] >= 90) | (df_depression["WBP16"] == 1) | (df_depression["WBP19"] == 1)).astype(int)

display(df_anxiety["Hypertension"].value_counts())
print("-"*40)
display(df_depression["Hypertension"].value_counts())

Hypertension
0    4204
1     933
Name: count, dtype: int64

----------------------------------------


Hypertension
0    4204
1     933
Name: count, dtype: int64

#### BMI Variable

In [25]:
for i in domain_groups.get("obesity"):
    parse(i)
    print("-"*40)

Name: Body Mass Index (BMI)
Value: Continuous variable
----------------------------------------


In [26]:
df_anxiety["Obesity"] = (df_anxiety["HA40"] >= 3000).astype(int)
df_depression["Obesity"] = (df_depression["HA40"] >= 3000).astype(int)

display(df_anxiety["Obesity"].value_counts())
print("-"*40)
display(df_depression["Obesity"].value_counts())

Obesity
0    4702
1     435
Name: count, dtype: int64

----------------------------------------


Obesity
0    4702
1     435
Name: count, dtype: int64

#### Combine Cardiometabolic Burden

In [27]:
parse("Cardiometabolic Burden")

Name: Cardiometabolic Burden
Categories: 
	0: None
	1: One burden
	2: Two burden
	3: Three burden


In [28]:
df_anxiety_new = pd.DataFrame()
df_depression_new = pd.DataFrame()

df_anxiety_new["Cardiometabolic Burden"] = df_anxiety[["Diabetes", "Hypertension", "Obesity"]].sum(axis=1)
df_depression_new["Cardiometabolic Burden"] = df_depression[["Diabetes", "Hypertension", "Obesity"]].sum(axis=1)

display(df_anxiety_new["Cardiometabolic Burden"].value_counts())
print("-"*40)
display(df_depression_new["Cardiometabolic Burden"].value_counts())

Cardiometabolic Burden
0    3648
1    1128
2     318
3      43
Name: count, dtype: int64

----------------------------------------


Cardiometabolic Burden
0    3648
1    1128
2     318
3      43
Name: count, dtype: int64

## Socioeconomic Status

In [29]:
parse("Socioeconomic Status")

Name: Socioeconomic Status (Wealth index combined)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest


In [30]:
df_anxiety_new["Socioeconomic Status"] = df_anxiety[["V190"]]
df_depression_new["Socioeconomic Status"] = df_depression[["V190"]]

display(df_anxiety_new["Socioeconomic Status"].value_counts())
print("-"*40)
display(df_depression_new["Socioeconomic Status"].value_counts())

Socioeconomic Status
5    1166
4    1076
2     994
3     992
1     909
Name: count, dtype: int64

----------------------------------------


Socioeconomic Status
5    1166
4    1076
2     994
3     992
1     909
Name: count, dtype: int64

## Anxiety and Depression

In [31]:
parse("Depression")
print("-"*40)
parse("Anxiety")

Name: PHQ-9 depression score (categorized)
Categories: 
	0: 0-4 (minimal)
	1: 5-9 (mild)
	2: 10-14 (moderate)
	3: 15-19 (moderately severe)
	4: 20-27 (severe)
----------------------------------------
Name: GAD-7 anxiety score (categorized)
Categories: 
	0: 0-4 (minimal)
	1: 5-9 (mild)
	2: 10-14 (moderate)
	3: 15-21 (severe)


In [32]:
def phq9(value: int) -> str:
    if value < 5:
        return 0
    elif value < 10:
        return 1
    elif value < 15:
        return 2
    elif value < 20:
        return 3
    else:
        return 4


def gad7(value: int) -> str:
    if value < 5:
        return 0
    elif value < 10:
        return 1
    elif value < 15:
        return 2
    else:
        return 3

In [33]:
df_depression_new["Depression"] = df_depression["MTH22"].apply(phq9)
df_anxiety_new["Anxiety"] = df_anxiety["MTH24"].apply(gad7)

display(df_depression_new["Depression"].value_counts())
print("-"*40)
display(df_anxiety_new["Anxiety"].value_counts())

Depression
0    3625
1    1253
2     190
3      61
4       8
Name: count, dtype: int64

----------------------------------------


Anxiety
0    3676
1    1213
2     196
3      52
Name: count, dtype: int64

In [34]:
display(df_depression_new.head())
display(df_anxiety_new.head())

,Cardiometabolic Burden,Socioeconomic Status,Depression
0,3,5,1
5,0,4,1
11,0,4,0
18,0,4,0
25,1,5,0


,Cardiometabolic Burden,Socioeconomic Status,Anxiety
0,3,5,1
5,0,4,1
11,0,4,0
18,0,4,0
25,1,5,0


## Recategorize Confounders

#### Current marital status

In [35]:
parse("S111A")
print("-"*40)
parse("Marital status")

Name: Current marital status
Categories: 
	1: Currently married
	2: Separated
	3: Deserted
	4: Divorced
	5: Widowed
----------------------------------------
Name: Current marital status
Categories: 
	1: Currently married
	2: Currently not married (Widowed / Divorced / Seperated / Deserted)


In [36]:
print("Available categories in the dataset")
print(df["S111A"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5]


In [37]:
def cms(value: int) -> int:
    if value == 1:
        return 1
    elif value < 6:
        return 2
    else:
        return np.nan

In [38]:
df_depression_new["Marital status"] = df_depression["S111A"].apply(cms)
df_anxiety_new["Marital status"] = df_anxiety["S111A"].apply(cms)

display(df_depression_new["Marital status"].value_counts())
print("-"*40)
display(df_anxiety_new["Marital status"].value_counts())

Marital status
1    4891
2     246
Name: count, dtype: int64

----------------------------------------


Marital status
1    4891
2     246
Name: count, dtype: int64

#### Education

In [39]:
parse("V106")
print("-"*40)
parse("Education")

Name: Highest educational level
Categories: 
	0: No education
	1: Primary
	2: Secondary
	3: Higher
----------------------------------------
Name: Highest educational level
Categories: 
	0: No education
	1: Primary
	2: Secondary
	3: Higher


In [40]:
print("Available categories in the dataset")
print(df["V106"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3]


In [41]:
df_depression_new["Education"] = df_depression["V106"]
df_anxiety_new["Education"] = df_anxiety["V106"]

display(df_depression_new["Education"].value_counts())
print("-"*40)
display(df_anxiety_new["Education"].value_counts())

Education
2    2246
1    1354
3     796
0     741
Name: count, dtype: int64

----------------------------------------


Education
2    2246
1    1354
3     796
0     741
Name: count, dtype: int64

#### Occupation

In [42]:
parse("V714")
print("-"*40)
parse("Occupation")

Name: Respondent currently working
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Respondent currently working
Categories: 
	0: No
	1: Yes


In [43]:
print("Available categories in the dataset")
print(df["V714"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0]


In [44]:
df_depression_new["Occupation"] = df_depression["V714"]
df_anxiety_new["Occupation"] = df_anxiety["V714"]

display(df_depression_new["Occupation"].value_counts())
print("-"*40)
display(df_anxiety_new["Occupation"].value_counts())

Occupation
0.0    3496
1.0    1641
Name: count, dtype: int64

----------------------------------------


Occupation
0.0    3496
1.0    1641
Name: count, dtype: int64

#### Husband/partner's occupation (grouped)

In [45]:
parse("V705")
print("-"*40)
parse("Partner occupation")

Name: Husband/partner's occupation (grouped)
Categories: 
	0: Not working
	1: Professional/technical/managerial
	2: Clerical
	3: Sales
	4: Agricultural - self employed
	5: Agricultural - employee
	6: Household and domestic
	7: Services
	8: Skilled manual
	9: Unskilled manual
	98: Don't know
----------------------------------------
Name: Husband/partner's occupation (grouped)
Categories: 
	1: Not working
	2: Working
	3: Currently not married (Widowed / Divorced / Seperated / Deserted)


In [46]:
print("Available categories in the dataset")
print(df["V705"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 98.0]


In [47]:
def po(value: int) -> int:
    if value == 0:
        return 1
    elif value < 10:
        return 2
    else:
        return np.nan

In [48]:
df_depression_new["Partner occupation"] = df_depression["V705"].apply(po)
df_anxiety_new["Partner occupation"] = df_anxiety["V705"].apply(po)

df_depression_new.loc[df_depression_new["Marital status"] == 2, "Partner occupation"] = 3
df_anxiety_new.loc[df_anxiety_new["Marital status"] == 2, "Partner occupation"] = 3

display(df_depression_new["Partner occupation"].value_counts())
print("-"*40)
display(df_anxiety_new["Partner occupation"].value_counts())

Partner occupation
2.0    4738
3.0     246
1.0     145
Name: count, dtype: int64

----------------------------------------


Partner occupation
2.0    4738
3.0     246
1.0     145
Name: count, dtype: int64

#### Age group

In [49]:
parse("V013")
print("-"*40)
parse("Age")

Name: Age in 5-year groups
Categories: 
	1: 15-19
	2: 20-24
	3: 25-29
	4: 30-34
	5: 35-39
	6: 40-44
	7: 45-49
----------------------------------------
Name: Age in 5-year groups
Categories: 
	1: 15-24
	2: 25-34
	3: 35-49


In [50]:
print("Available categories in the dataset")
print(df["V013"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7]


In [51]:
def age(value: int) -> int:
    if value in [1, 2]:
        return 1
    elif value in [3, 4]:
        return 2
    elif value < 8:
        return 3
    else:
        return np.nan

In [52]:
df_depression_new["Age"] = df_depression["V013"].apply(age)
df_anxiety_new["Age"] = df_anxiety["V013"].apply(age)

display(df_depression_new["Age"].value_counts())
print("-"*40)
display(df_anxiety_new["Age"].value_counts())

Age
3    2193
2    1797
1    1147
Name: count, dtype: int64

----------------------------------------


Age
3    2193
2    1797
1    1147
Name: count, dtype: int64

#### Residence (Division)

In [53]:
parse("V024")
print("-"*40)
parse("Division")

Name: Division
Categories: 
	1: Barishal
	2: Chattogram
	3: Dhaka
	4: Khulna
	5: Mymensingh
	6: Rajshahi
	7: Rangpur
	8: Sylhet
----------------------------------------
Name: Division
Categories: 
	1: Barishal
	2: Chattogram
	3: Dhaka
	4: Khulna
	5: Mymensingh
	6: Rajshahi
	7: Rangpur
	8: Sylhet


In [54]:
print("Available categories in the dataset")
print(df["V024"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7, 8]


In [55]:
df_depression_new["Division"] = df_depression["V024"]
df_anxiety_new["Division"] = df_anxiety["V024"]

display(df_depression_new["Division"].value_counts())
print("-"*40)
display(df_anxiety_new["Division"].value_counts())

Division
2    780
3    743
4    658
6    647
7    605
8    579
1    568
5    557
Name: count, dtype: int64

----------------------------------------


Division
2    780
3    743
4    658
6    647
7    605
8    579
1    568
5    557
Name: count, dtype: int64

#### Residence (Urban/Rural)

In [56]:
parse("V025")
print("-"*40)
parse("Residence")

Name: Type of place of residence
Categories: 
	1: Urban
	2: Rural
----------------------------------------
Name: Type of place of residence
Categories: 
	1: Urban
	2: Rural


In [57]:
print("Available categories in the dataset")
print(df["V025"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2]


In [58]:
df_depression_new["Residence"] = df_depression["V025"]
df_anxiety_new["Residence"] = df_anxiety["V025"]

display(df_depression_new["Residence"].value_counts())
print("-"*40)
display(df_anxiety_new["Residence"].value_counts())

Residence
2    3324
1    1813
Name: count, dtype: int64

----------------------------------------


Residence
2    3324
1    1813
Name: count, dtype: int64

#### Religion

In [59]:
parse("V130")
print("-"*40)
parse("Religion")

Name: Religion
Categories: 
	1: Islam
	2: Hindu
	3: Christian
	4: Other
----------------------------------------
Name: Religion
Categories: 
	1: Islam
	2: Others


In [60]:
print("Available categories in the dataset")
print(df["V130"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 96]


In [61]:
def religion(value: int) -> int:
    if value == 1:
        return 1
    elif value < 5:
        return 2
    else:
        return np.nan

In [62]:
df_depression_new["Religion"] = df_depression["V130"].apply(religion)
df_anxiety_new["Religion"] = df_anxiety["V130"].apply(religion)

display(df_depression_new["Religion"].value_counts())
print("-"*40)
display(df_anxiety_new["Religion"].value_counts())

Religion
1.0    4621
2.0     515
Name: count, dtype: int64

----------------------------------------


Religion
1.0    4621
2.0     515
Name: count, dtype: int64

#### Total children ever born

In [63]:
parse("V201")
print("-"*40)
parse("Children")

Name: Total children ever born
Value: Continuous variable
----------------------------------------
Name: Total children ever born
Categories: 
	0: No children
	1: 1
	2: 2
	3: 3
	4: 4 or more


In [64]:
print("Available categories in the dataset")
print(df["V201"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [65]:
def children(value: int) -> int:
    if value in range(0, 4):
        return value
    elif value < 12:
        return 4
    else:
        return np.nan

In [66]:
df_depression_new["Children"] = df_depression["V201"].apply(children)
df_anxiety_new["Children"] = df_anxiety["V201"].apply(children)

display(df_depression_new["Children"].value_counts())
print("-"*40)
display(df_anxiety_new["Children"].value_counts())

Children
2    1671
1    1123
3    1073
4     814
0     456
Name: count, dtype: int64

----------------------------------------


Children
2    1671
1    1123
3    1073
4     814
0     456
Name: count, dtype: int64

#### Number of household members

In [67]:
parse("V136")
print("-"*40)
parse("Family size")

Name: Number of household members
Value: Continuous variable
----------------------------------------
Name: Number of household members
Categories: 
	1: Less than 5
	2: 5 or more


In [68]:
print("Available categories in the dataset")
print(df["V136"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25]


In [69]:
def size(value: int) -> int:
    if value in range(0, 5):
        return 1
    elif value < 26:
        return 2
    else:
        return np.nan

In [70]:
df_depression_new["Family size"] = df_depression["V136"].apply(size)
df_anxiety_new["Family size"] = df_anxiety["V136"].apply(size)

display(df_depression_new["Family size"].value_counts())
print("-"*40)
display(df_anxiety_new["Family size"].value_counts())

Family size
2    2710
1    2427
Name: count, dtype: int64

----------------------------------------


Family size
2    2710
1    2427
Name: count, dtype: int64

#### Autonomy in household decision

In [71]:
for i in domain_groups.get("autonomy"):
    parse(i)
    print("-"*40)
parse("Autonomy")

Name: Person who decides on respondent's health care
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
----------------------------------------
Name: Person who decides on large household purchases
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
----------------------------------------
Name: Person who decides on visits to family/relatives
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
----------------------------------------
Name: Person who decides what to do with money husband earns
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: S

In [72]:
print("Available categories in the dataset")
for i in domain_groups.get("autonomy"):
    print(df[i].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1.0, 2.0, 4.0, 5.0, 6.0]
[1.0, 2.0, 4.0, 5.0, 6.0]
[1.0, 2.0, 4.0, 5.0, 6.0]
[1.0, 2.0, 4.0, 6.0, 7.0]


In [73]:
def autonomy(value: int) -> int:
    if value in [1, 2, 3]:
        return 1
    elif value in [4, 5, 6]:
        return 0
    else:
        return np.nan

In [74]:
df_depression_new["Autonomy"] = df_depression[domain_groups.get("autonomy")].map(autonomy).sum(axis=1)
df_anxiety_new["Autonomy"] = df_anxiety[domain_groups.get("autonomy")].map(autonomy).sum(axis=1)

display(df_depression_new["Autonomy"].value_counts())
print("-"*40)
display(df_anxiety_new["Autonomy"].value_counts())

Autonomy
4.0    2505
0.0     865
3.0     750
2.0     526
1.0     491
Name: count, dtype: int64

----------------------------------------


Autonomy
4.0    2505
0.0     865
3.0     750
2.0     526
1.0     491
Name: count, dtype: int64

#### Physical/Sexual/Emotional Abuse

In [75]:
for i in domain_groups.get("ipv_attitudes"):
    parse(i)
print("-"*40)
parse("Abuse")

Name: Wife beating justified if she goes out without telling husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she neglects the children
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she argues with husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she refuses to have sex with husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she burns the food
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
----------------------------------------
Name: Physical/Sexual/Emotional Abuse
Categories: 
	0: No
	1: Yes


In [76]:
print("Available categories in the dataset")
for i in domain_groups.get("ipv_attitudes"):
    print(df[i].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]


In [77]:
def ipv_attitudes(value: int) -> int:
    if value in [0, 1]:
        return value
    else:
        return np.nan

In [78]:
df_depression_new["Abuse"] = df_depression[domain_groups.get("ipv_attitudes")].map(ipv_attitudes).any(axis=1).astype(int)
df_anxiety_new["Abuse"] = df_anxiety[domain_groups.get("ipv_attitudes")].map(ipv_attitudes).any(axis=1).astype(int)

display(df_depression_new["Abuse"].value_counts())
print("-"*40)
display(df_anxiety_new["Abuse"].value_counts())

Abuse
0    4461
1     676
Name: count, dtype: int64

----------------------------------------


Abuse
0    4461
1     676
Name: count, dtype: int64

#### Health insurance

In [79]:
parse("V481")
print("-"*40)
parse("Insurance")

Name: Covered by health insurance
Categories: 
	0: No
	1: Yes
	9: Missing
----------------------------------------
Name: Covered by health insurance
Categories: 
	0: No
	1: Yes


In [80]:
print("Available categories in the dataset")
print(df["V481"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0]


In [81]:
def insurance(value: int) -> int:
    if value in [0, 1]:
        return value
    else:
        return np.nan

In [82]:
df_depression_new["Insurance"] = df_depression["V481"].apply(insurance)
df_anxiety_new["Insurance"] = df_anxiety["V481"].apply(insurance)

display(df_depression_new["Insurance"].value_counts())
print("-"*40)
display(df_anxiety_new["Insurance"].value_counts())

Insurance
0.0    5121
1.0      16
Name: count, dtype: int64

----------------------------------------


Insurance
0.0    5121
1.0      16
Name: count, dtype: int64

#### Use of Internet

In [83]:
parse("V171A")
print("-"*40)
parse("Internet")

Name: Use of internet
Categories: 
	0: Never
	1: Yes, last 12 months
	2: Yes, before last 12 months
	3: Yes, can't establish when
	9: Missing
----------------------------------------
Name: Use of internet
Categories: 
	0: Never
	1: Occasionally
	2: Yes


In [84]:
print("Available categories in the dataset")
print(df["V171A"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 2.0]


In [85]:
def internet(value: int) -> int:
    if value == 0:
        return 0
    elif value in [2, 3]:
        return 1
    elif value == 1:
        return 2
    else:
        return np.nan

In [86]:
df_depression_new["Internet"] = df_depression["V171A"].apply(internet)
df_anxiety_new["Internet"] = df_anxiety["V171A"].apply(internet)

display(df_depression_new["Internet"].value_counts())
print("-"*40)
display(df_anxiety_new["Internet"].value_counts())

Internet
0    3673
2    1439
1      25
Name: count, dtype: int64

----------------------------------------


Internet
0    3673
2    1439
1      25
Name: count, dtype: int64

#### Current contraceptive use by method type

In [87]:
parse("V313")
print("-"*40)
parse("Contraceptive")

Name: Current use by method type (simplified/collapsed version)
Categories: 
	0: No method
	1: Folkloric method
	2: Traditional method
	3: Modern method
	9: Missing
----------------------------------------
Name: Current contraceptive use by method type (simplified/collapsed version)
Categories: 
	0: No method
	1: Folkloric method
	2: Traditional method
	3: Modern method


In [88]:
print("Available categories in the dataset")
print(df["V313"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 2.0, 3.0]


In [89]:
def contraceptive(value: int) -> int:
    if value in range(0, 4):
        return value
    else:
        return np.nan

In [90]:
df_depression_new["Contraceptive"] = df_depression["V313"].apply(contraceptive)
df_anxiety_new["Contraceptive"] = df_anxiety["V313"].apply(contraceptive)

display(df_depression_new["Contraceptive"].value_counts())
print("-"*40)
display(df_anxiety_new["Contraceptive"].value_counts())

Contraceptive
3.0    2691
0.0    1924
2.0     517
1.0       5
Name: count, dtype: int64

----------------------------------------


Contraceptive
3.0    2691
0.0    1924
2.0     517
1.0       5
Name: count, dtype: int64

#### Miscarriage/Abortion

In [91]:
parse("V228")
print("-"*40)
parse("Abortion")

Name: Ever had a terminated pregnancy
Categories: 
	0: No
	1: Yes
	9: Missing
----------------------------------------
Name: Ever had a terminated pregnancy
Categories: 
	0: No
	1: Yes


In [92]:
print("Available categories in the dataset")
print(df["V228"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1]


In [93]:
def abortion(value: int) -> int:
    if value in range(0, 2):
        return value
    else:
        return np.nan

In [94]:
df_depression_new["Abortion"] = df_depression["V228"].apply(abortion)
df_anxiety_new["Abortion"] = df_anxiety["V228"].apply(abortion)

display(df_depression_new["Abortion"].value_counts())
print("-"*40)
display(df_anxiety_new["Abortion"].value_counts())

Abortion
0    3810
1    1327
Name: count, dtype: int64

----------------------------------------


Abortion
0    3810
1    1327
Name: count, dtype: int64

#### Currently pregnant

In [95]:
parse("V213")
print("-"*40)
parse("Pregnant")

Name: Currently pregnant
Categories: 
	0: No or unsure
	1: Yes
	9: Missing
----------------------------------------
Name: Currently pregnant
Categories: 
	0: No or unsure
	1: Yes


In [96]:
print("Available categories in the dataset")
print(df["V213"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1]


In [97]:
def pregnant(value: int) -> int:
    if value in range(0, 2):
        return value
    else:
        return np.nan

In [98]:
df_depression_new["Pregnant"] = df_depression["V213"].apply(pregnant)
df_anxiety_new["Pregnant"] = df_anxiety["V213"].apply(pregnant)

display(df_depression_new["Pregnant"].value_counts())
print("-"*40)
display(df_anxiety_new["Pregnant"].value_counts())

Pregnant
0    4851
1     286
Name: count, dtype: int64

----------------------------------------


Pregnant
0    4851
1     286
Name: count, dtype: int64

#### In menopause

In [99]:
parse("V226")
print("-"*40)
parse("Menopause")

Name: Time since last period (comp) (months)
Categories: 
	Continuous: 0:400
	994: In menopause
	995: Before last pregnancy
	996: Never menstruated
	997: Inconsistent
	998: Don't know
	999: Missing
----------------------------------------
Name: In menopause
Categories: 
	0: No
	1: Yes


In [100]:
print("Available categories in the dataset")
print(df["V226"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 27, 28, 30, 31, 34, 36, 41, 42, 45, 46, 48, 56, 58, 60, 71, 72, 84, 96, 108, 120, 132, 144, 156, 168, 180, 192, 204, 216, 228, 240, 276, 994, 995, 996, 997]


In [101]:
def menopause(value: int) -> int:
    if value == 994:
        return 1
    elif value < 998:
        return 0
    else:
        return np.nan

In [102]:
df_depression_new["Menopause"] = df_depression["V226"].apply(menopause)
df_anxiety_new["Menopause"] = df_anxiety["V226"].apply(menopause)

display(df_depression_new["Menopause"].value_counts())
print("-"*40)
display(df_anxiety_new["Menopause"].value_counts())

Menopause
0    4826
1     311
Name: count, dtype: int64

----------------------------------------


Menopause
0    4826
1     311
Name: count, dtype: int64

#### Recent sexual activity

In [103]:
parse("V536")
print("-"*40)
parse("Sexual activity")

Name: Recent sexual activity
Categories: 
	0: Never had sex
	1: Active in last 4 weeks
	2: Not active in last 4 weeks - postpartum abstinence
	3: Not active in last 4 weeks - not postpartum abstinence
	9: Missing
----------------------------------------
Name: Recent sexual activity
Categories: 
	0: Not Active
	1: Active


In [104]:
print("Available categories in the dataset")
print(df["V536"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 2.0, 3.0]


In [105]:
def sexual_activity(value: int) -> int:
    if value == 1:
        return 1
    elif value in [0, 2, 3]:
        return 0
    else:
        return np.nan

In [106]:
df_depression_new["Sexual activity"] = df_depression["V536"].apply(sexual_activity)
df_anxiety_new["Sexual activity"] = df_anxiety["V536"].apply(sexual_activity)

display(df_depression_new["Sexual activity"].value_counts())
print("-"*40)
display(df_anxiety_new["Sexual activity"].value_counts())

Sexual activity
1    3942
0    1195
Name: count, dtype: int64

----------------------------------------


Sexual activity
1    3942
0    1195
Name: count, dtype: int64

#### Postpartum abstinence

In [107]:
parse("V536")
print("-"*40)
parse("postpartum")

Name: Recent sexual activity
Categories: 
	0: Never had sex
	1: Active in last 4 weeks
	2: Not active in last 4 weeks - postpartum abstinence
	3: Not active in last 4 weeks - not postpartum abstinence
	9: Missing
----------------------------------------
Variable not found in the recode list!


In [108]:
print("Available categories in the dataset")
print(df["V536"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 2.0, 3.0]


In [109]:
def postpartum(value: int) -> int:
    if value == 2:
        return 1
    elif value in [0, 1, 3]:
        return 0
    else:
        return np.nan

In [110]:
df_depression_new["Postpartum"] = df_depression["V536"].apply(postpartum)
df_anxiety_new["Postpartum"] = df_anxiety["V536"].apply(postpartum)

display(df_depression_new["Postpartum"].value_counts())
print("-"*40)
display(df_anxiety_new["Postpartum"].value_counts())

Postpartum
0    5015
1     122
Name: count, dtype: int64

----------------------------------------


Postpartum
0    5015
1     122
Name: count, dtype: int64

## Sampling Design

In [111]:
(df['V001'] == df['V021']).mean()

np.float64(1.0)

**Since the mean is 1, hence `V001` and `V021` are identical. We would use `V021` instead of `V001`**

In [ ]:
df_depression_new["CASEID"] = df_depression["CASEID"]
df_anxiety_new["CASEID"] = df_anxiety["CASEID"]

# df_depression_new["Cluster number"] = df_depression["V001"]
# df_anxiety_new["Cluster number"] = df_anxiety["V001"]

df_depression_new["Sampling weight"] = df_depression["V005"] / 1000000
df_anxiety_new["Sampling weight"] = df_anxiety["V005"] / 1000000

df_depression_new["PSU"] = df_depression["V021"]
df_anxiety_new["PSU"] = df_anxiety["V021"]

df_depression_new["Stratum"] = df_depression["V022"]
df_anxiety_new["Stratum"] = df_anxiety["V022"]

In [113]:
df_depression_new.to_csv('../resources/depression_dataset.csv', index=False)
df_anxiety_new.to_csv('../resources/anxiety_dataset.csv', index=False)